# Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
warnings.filterwarnings("ignore", ".*dubious year.*")
warnings.filterwarnings(
    "ignore", "Tried to get polar motions for times after IERS data is valid.*"
)

In [ ]:
import numpy as np
from astropy import units as u
from astropy.coordinates import (
    SkyCoord,
)
from ligo.skymap import plot  # noqa: F401
from m4opt import fov
from m4opt.missions import uvex as mission
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from regions import CircleSkyRegion, Region
from tqdm.auto import tqdm

# FOV

For any given target sky position, UVEX's roll angle rotates by 360° throughout the year. Plot circle inscribed within the FOV.

In [ ]:
inscribed_fov = CircleSkyRegion(SkyCoord(0 * u.deg, 0 * u.deg), mission.fov.width / 2)

In [ ]:
width, height = plt.rcParams["figure.figsize"]
fig = plt.figure(figsize=(width, width))
ax = fig.add_subplot(projection="astro degrees zoom", radius=3 * u.deg)

ax.add_patch(inscribed_fov.to_pixel(ax.wcs).as_artist())
ax.grid()
for key in ["ra", "dec"]:
    ax.coords[key].set_auto_axislabel(False)

blit = []
rolls = np.linspace(0, 90, endpoint=False) * u.deg
regions = fov.footprint(mission.fov, SkyCoord(0 * u.deg, 0 * u.deg), rolls)

with tqdm(total=len(regions)) as progress:

    def animate(region: Region):
        for artist in blit:
            artist.remove()
        del blit[:]
        blit.append(ax.add_patch(region.to_pixel(ax.wcs).as_artist(linewidth=2)))
        progress.update()
        return blit

    FuncAnimation(fig, animate, regions, blit=True, interval=50).save(
        "../visualizations/fov.mp4"
    )

Save the FOV footprint and the inscribed circle as DS9 regions.

In [ ]:
mission.fov.copy(meta={"text": "UVEX field of view"}).write(
    "../fov.ds9", overwrite=True
)
inscribed_fov.copy(meta={"text": "Circle inscribed within UVEX field of view"}).write(
    "../fov-inscribed-circle.ds9", overwrite=True
)